# Track 4: Aligned Response Refinement with Direct Preference Optimization (DPO)

This notebook demonstrates how to align a model with preferred human feedback using Direct Preference Optimization (DPO) and `DPOTrainer` from Hugging Face `trl`. We use **Unsloth** for memory-efficient and fast LoRA training of the **Qwen2.5-3B-Instruct** model.

## 1. Setup Environment and Imports
We load our dependencies and identify the compute device capability to determine if bfloat16 is supported.

In [1]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import os
import torch
import warnings
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import DPOTrainer, DPOConfig

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)
print(f"Using device: cuda | Dtype: {compute_dtype}")

[unsloth.import_fixes|WARNING]Unsloth: Detected broken vLLM binary extension; disabling vLLM imports and continuing import.
Please reinstall via `uv pip install unsloth vllm torchvision torchaudio --torch-backend=auto`.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0721 16:58:34.730000 351704 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0721 16:58:34.744000 351704 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/home/lmassaron/code/sft-examples/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:1200: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


Using device: cuda | Dtype: torch.bfloat16


## 2. Load Model & Enable PEFT
We load `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` using Unsloth's optimized 4-bit precision to fit within small VRAM budgets, and attach LoRA adapters to all projection layers.

In [2]:
MODEL_ID = "unsloth/Qwen2.5-3B-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

# Apply ChatML template to the tokenizer since Qwen2.5 base model does not have a default chat template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model and PEFT adapters loaded successfully.")

==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.14.1. vLLM: 0.19.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[unsloth.chat_templates|WARNING]Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.


Unsloth 2026.7.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model and PEFT adapters loaded successfully.


## 3. Load and Format Dataset (Cleaned Orca DPO Pairs)
We load the `argilla/distilabel-intel-orca-dpo-pairs` dataset from Hugging Face, which is a cleaned, high-quality version of the original Intel Orca dataset with reduced label noise. We shuffle it, select a subset of 250 training examples and 50 validation examples, and map the inputs to a standard DPO format where the user prompt is compiled using the model's native chat template.

In [3]:
dataset = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train")
shuffled = dataset.shuffle(seed=42)
train_ds = shuffled.select(range(250))
eval_ds = shuffled.select(range(250, 300))


def format_dpo_example(example):
    messages = [{"role": "user", "content": example["input"]}]
    prompt_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return {
        "prompt": prompt_str,
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }


train_mapped = train_ds.map(format_dpo_example, remove_columns=train_ds.column_names)
eval_mapped = eval_ds.map(format_dpo_example, remove_columns=eval_ds.column_names)
print("Prompt Preview:\n", train_mapped[0]["prompt"])
print("Chosen Preview:\n", train_mapped[0]["chosen"])
print("Rejected Preview:\n", train_mapped[0]["rejected"])

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Prompt Preview:
 <|im_start|>user
This is some data: CBS PLAY-BY-PLAY Chris Schenkel (first half) and Ray Scott (second half); 1962 NETWORK CBS.

Generate a detailed description of this data<|im_end|>
<|im_start|>assistant

Chosen Preview:
 Okay, imagine you are watching a fun game on TV with your family. In this case, the game happened in 1962. Now, on TV, there are people who talk to us and tell us what is happening in the game. They help us understand the game better, just like how I'm helping you understand things right now.

In this data, there are two people who talked about the game in 1962. The first person, Chris Schenkel, talked about the game in the first half. The second person, Ray Scott, talked about the game in the second half. Both of them worked for a big TV company called CBS. So, this sentence is just telling us who talked about the game on TV and when they did it.
Rejected Preview:
  OH MY GOSH, YOU WANT TO KNOW ABOUT THIS SUPER COOL DATA?! 😍

Okay, so let me tell y

## 3.5. Evaluate Base Model before DPO Training

To perform a fair comparison of how DPO training aligns the model's response style, we evaluate the base (unaligned) model on a set of test prompts first. We store these responses to compare them with the DPO-aligned model later.

In [4]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "Explain why the sky is blue in one concise sentence.",
    "What is a metric ton?",
    "Why did the Roman Empire fall? Explain the primary contributing factors.",
    "What is the difference between existentialism and nihilism?",
]

model.generation_config.max_length = None  # Suppress the max_length warning
base_responses = {}
for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=100,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    base_responses[prompt] = tokenizer.decode(
        outputs[0][inputs.shape[1] :], skip_special_tokens=True
    ).strip()
    print(f"Prompt: {prompt}\nBase Response: {base_responses[prompt]}\n" + "-" * 50)

# Restore training mode for the model
FastLanguageModel.for_training(model)

Prompt: Explain why the sky is blue in one concise sentence.
Base Response: The sky appears blue because of Rayleigh scattering, which causes shorter blue wavelengths to scatter more than longer red wavelengths, making the sky appear blue.
--------------------------------------------------


Prompt: What is a metric ton?
Base Response: A metric ton is a unit of measurement used to measure the weight of objects. It is equal to 1,000 kilograms or 2,204.62 pounds. It is commonly used in the metric system and is often used to measure the weight of goods, such as cargo or bulk materials.
--------------------------------------------------


Prompt: Why did the Roman Empire fall? Explain the primary contributing factors.
Base Response: The fall of the Roman Empire is a complex and multifaceted historical event that can be attributed to a combination of internal and external factors. Here are some of the primary contributing factors:

1. Political instability: The Roman Empire was plagued by political instability, corruption, and power struggles. Emperors often appointed their own relatives and friends to key positions, leading to a lack of stability and a decline in the quality of governance.

2. Economic decline: The Roman Empire experienced a significant economic decline in
--------------------------------------------------


Prompt: What is the difference between existentialism and nihilism?
Base Response: Existentialism and nihilism are two different philosophical perspectives that deal with the meaning of life and the human condition. Existentialism emphasizes the individual's freedom and responsibility to create their own meaning and purpose in life, while nihilism denies the existence of any objective meaning or value in life. In other words, existentialism focuses on the individual's subjective experience and the search for personal fulfillment, while nihilism denies the possibility of finding any objective meaning or purpose in life.
--------------------------------------------------


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.L

## 4. Run Aligned DPO Fine-Tuning
We run Unsloth's patched version of the TRL `DPOTrainer` (`PatchDPOTrainer()`). We run training for 120 steps with a learning rate of 5e-6, using cosine annealing decay.

In [5]:
PatchDPOTrainer()


training_args = DPOConfig(
    output_dir="qwen2.5-3b-dpo-output",
    beta=0.1,
    max_length=1024,
    max_prompt_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=120,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=20,
    report_to="none",
)


trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    processing_class=tokenizer,
)


model.config.use_cache = False

trainer.train()


model.save_pretrained("qwen2.5-3b-dpo-adapter")

tokenizer.save_pretrained("qwen2.5-3b-dpo-adapter")

print("DPO adapter successfully saved!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Extracting prompt in train dataset (num_proc=24):   0%|          | 0/250 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=24):   0%|          | 0/250 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=24):   0%|          | 0/250 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=24):   0%|          | 0/50 [00:00<?, ? examples/s]

...

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-80/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-100/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-120/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-adapter/tokenizer_config.json.


DPO adapter successfully saved!


## 5. Evaluation and Inference Comparison
We put the model back in inference mode and query it with our set of test prompts to see the side-by-side style change between the base model and the DPO-aligned model.

In [6]:
FastLanguageModel.for_inference(model)

model.generation_config.max_length = None  # Suppress the max_length warning
for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=100,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    dpo_response = tokenizer.decode(
        outputs[0][inputs.shape[1] :], skip_special_tokens=True
    ).strip()

    print(f"Prompt: {prompt}")
    print(f"\033[91mBefore DPO (Base Model):\033[0m\n{base_responses[prompt]}")
    print(f"\033[92mAfter DPO (Aligned Model):\033[0m\n{dpo_response}")
    print("-" * 80 + "\n")

Prompt: Explain why the sky is blue in one concise sentence.
Before DPO (Base Model):
The sky appears blue because of Rayleigh scattering, which causes shorter blue wavelengths to scatter more than longer red wavelengths, making the sky appear blue.
After DPO (Aligned Model):
The sky appears blue because of Rayleigh scattering, which causes shorter blue wavelengths to scatter more than longer red wavelengths, making the sky appear blue.
--------------------------------------------------------------------------------



Prompt: What is a metric ton?
Before DPO (Base Model):
A metric ton is a unit of measurement used to measure the weight of objects. It is equal to 1,000 kilograms or 2,204.62 pounds. It is commonly used in the metric system and is often used to measure the weight of goods, such as cargo or bulk materials.
After DPO (Aligned Model):
A metric ton is a unit of measurement used to measure the weight of objects. It is equal to 1,000 kilograms or 2,204.62 pounds. It is commonly used in industries, agriculture, and transportation to measure the weight of goods, vehicles, and other heavy items.
--------------------------------------------------------------------------------



Prompt: Why did the Roman Empire fall? Explain the primary contributing factors.
Before DPO (Base Model):
The fall of the Roman Empire is a complex and multifaceted historical event that can be attributed to a combination of internal and external factors. Here are some of the primary contributing factors:

1. Political instability: The Roman Empire was plagued by political instability, corruption, and power struggles. Emperors often appointed their own relatives and friends to key positions, leading to a lack of stability and a decline in the quality of governance.

2. Economic decline: The Roman Empire experienced a significant economic decline in
After DPO (Aligned Model):
The fall of the Roman Empire is a complex and multifaceted historical event that can be attributed to a combination of internal and external factors. Here are some of the primary contributing factors:

1. Political instability: The Roman Empire experienced a series of civil wars and power struggles that weakened th

Prompt: What is the difference between existentialism and nihilism?
Before DPO (Base Model):
Existentialism and nihilism are two different philosophical perspectives that deal with the meaning of life and the human condition. Existentialism emphasizes the individual's freedom and responsibility to create their own meaning and purpose in life, while nihilism denies the existence of any objective meaning or value in life. In other words, existentialism focuses on the individual's subjective experience and the search for personal fulfillment, while nihilism denies the possibility of finding any objective meaning or purpose in life.
After DPO (Aligned Model):
Existentialism and nihilism are two distinct philosophical perspectives that differ in their views on the nature of existence, human freedom, and the meaning of life. Here's a brief overview of each:

1. **Existentialism**:
   - **Definition**: Existentialism is a philosophical movement that emphasizes individual freedom, choice, and 